In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from numba import njit
import time

#\definecolor{Red}{RGB}{ 159, 36, 48 }
#\definecolor{LBlue}{RGB}{ badcff }
#\definecolor{DBlue}{HTML}{132C53} 
#\definecolor{gray}{HTML}{708090}

@njit
def simular_modelo_votante(tamanho, passos_tempo):
    """
    Simula o Modelo do Votante 2D.
    Usamos @njit (Numba) para deixar o código em C/C++ veloz,
    pois simulações de Monte Carlo no Python puro são muito lentas.
    """
    # Cria a rede inicial com 50% preto (0) e 50% branco (1)
    rede = np.random.randint(0, 2, size=(tamanho, tamanho))
    
    # Número total de sítios (N = L*L).
    # 1 passo de tempo (sweep) equivale a tentar atualizar N sítios.
    N = tamanho * tamanho
    
    # Vetores de direção para os vizinhos (Cima, Baixo, Esquerda, Direita)
    dx = np.array([0, 0, -1, 1])
    dy = np.array([-1, 1, 0, 0])
    
    for t in range(passos_tempo):
        for _ in range(N):
            # 1. Sorteia um indivíduo aleatório na rede
            x = np.random.randint(tamanho)
            y = np.random.randint(tamanho)
            
            # 2. Sorteia um dos 4 vizinhos
            direcao = np.random.randint(4)
            
            # Calculamos a posição do vizinho com % tamanho para
            # garantir condições de contorno periódicas (a borda direita 
            # conecta com a esquerda, como num "pac-man" ou toro).
            nx = (x + dx[direcao]) % tamanho
            ny = (y + dy[direcao]) % tamanho
            
            # 3. Regra do Votante: O indivíduo copia a opinião do vizinho
            rede[x, y] = rede[nx, ny]
            
    return rede

# ==========================================
# Configurações da Simulação
# ==========================================
L = 1024          # Tamanho da rede (512x512). Maior = mais bonito, mas mais pesado.
sweeps = 1000     # Tempo de evolução. Quanto maior, maiores os domínios fractais.

print(f"Simulando rede {L}x{L} por {sweeps} passos...")
inicio = time.time()

# Executa a simulação
rede_final = simular_modelo_votante(L, sweeps)

fim = time.time()
print(f"Simulação concluída em {fim - inicio:.2f} segundos!")

# ==========================================
# Plotando a Imagem
# ==========================================
plt.figure(figsize=(8, 8), facecolor='#badcff')

minhas_cores = ['#badcff', '#132C53']

# 2. Criamos o colormap customizado
meu_cmap = ListedColormap(minhas_cores)

# 3. Plotamos usando o seu cmap
plt.imshow(rede_final, cmap=meu_cmap, interpolation='nearest')

plt.axis('off') # Remove os eixos para focar só na textura fractal

plt.tight_layout()
plt.savefig('fractal-dyn-voter.svg')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Circle
from numba import njit
import time

# ==========================================
# 1. Função de Simulação (agora recebe a rede pronta)
# ==========================================
@njit
def simular_modelo_votante(rede_inicial, passos_tempo):
    # Fazemos uma cópia para não alterar a matriz original
    rede = rede_inicial.copy()
    
    tamanho = rede.shape[0]
    N = tamanho * tamanho
    dx = np.array([0, 0, -1, 1])
    dy = np.array([-1, 1, 0, 0])
    
    for t in range(passos_tempo):
        for _ in range(N):
            x = np.random.randint(tamanho)
            y = np.random.randint(tamanho)
            direcao = np.random.randint(4)
            nx = (x + dx[direcao]) % tamanho
            ny = (y + dy[direcao]) % tamanho
            rede[x, y] = rede[nx, ny]
            
    return rede

# ==========================================
# 2. Criando a Condição Inicial (O Círculo)
# ==========================================
L = 1024
raio_inicial = 300
centro = L // 2

# Criamos uma malha de coordenadas (X e Y)
Y, X = np.ogrid[:L, :L]

# Calculamos a distância ao quadrado de cada ponto até o centro (Pitágoras)
distancia_quadrada = (X - centro)**2 + (Y - centro)**2

# A mágica do Numpy: cria a matriz onde é 1 se estiver dentro do raio, e 0 se fora.
rede_bolha = (distancia_quadrada <= raio_inicial**2).astype(np.int8)

# ==========================================
# 3. Executando a Simulação
# ==========================================
sweeps = 5000 # Tempo de evolução (tente 10, 50, 150...)

print(f"Simulando bolha por {sweeps} passos...")
inicio = time.time()
rede_final = simular_modelo_votante(rede_bolha, sweeps)
fim = time.time()
print(f"Simulação concluída em {fim - inicio:.2f} segundos!")

# ==========================================
# 4. Plotando a Imagem
# ==========================================
plt.figure(figsize=(8, 8), facecolor='white')
ax = plt.gca() # Pegamos o eixo atual para poder desenhar o círculo base

# O seu colormap customizado
minhas_cores = ['#badcff', '#132C53']
meu_cmap = ListedColormap(minhas_cores)

# Plota a rede evoluída
plt.imshow(rede_final, cmap=meu_cmap, interpolation='nearest')

# Adiciona uma linha de contorno mostrando onde o círculo estava no t=0
# Igualzinho à "thin circle" mencionada na legenda do artigo!
circulo_referencia = Circle((centro, centro), raio_inicial, 
                            color='gray', fill=False, linewidth=2, linestyle='--')
ax.add_patch(circulo_referencia)

plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
!pip install numba